# PrecisionSkin - Eczema: Draft boxes, Review, Add to the dataset

The eczema class is the weakest (324 boxed photos). Your classifier folder has hundreds more facial eczema images **without boxes**. This notebook adds them
the honest way:

1. **Draft** - the trained detector proposes eczema boxes (machine-made, *unverified*).
2. **Review** - people open each image in a box-editing tool, delete wrong boxes, add missed ones and **save**. Nothing is used until a person has saved it.
3. **Measure** - how good were the drafts? (precision / recall of drafts against the corrected boxes - a number for your paper)
4. **Add** - the corrected images become a new source folder `eczema_corrected.yolov8`; you re-run the audit and the export notebook (`DATASET_VERSION = "v2"`).

**Honesty rules built in**
- Every box in this folder is labelled `machine-drafted, human-corrected` (not "human-drawn").
- Images from the classifier's **test split are not drafted** (they stay out of training, for the later Grad-CAM++ vs detector comparison).
- Images that already have a boxed twin in your Roboflow data are skipped (they already have human boxes).
- The detector's own **test set is never touched**; evaluation stays on the original human-drawn boxes.
- Originals are only read. Copies for review go to a Windows folder; the final source folder is created only when you set `APPLY_FINAL = True`.

Run this notebook **after** the training finishes (it needs the GPU and RAM). Run the cells in order; Stage 2 and Stage 4-6 can be re-run any time.

## Stage 0 - Config  (edit this cell only)

In [ ]:
import os, json, shutil, time, random, collections
from pathlib import Path
import numpy as np, pandas as pd

HOME = Path.home() / "PrecisionSkinV1"

# --- inputs -----------------------------------------------------------------------------------------------------
CANDIDATES_CSV = Path("/mnt/c/Users/Bautista/source/repos/SkinDevApp/eczema_facial_candidates.csv")   # made earlier (649 facial eczema images)
MODEL_PATH = HOME / "detector_work" / "runs" / "detector_v1_yolov8n_seed42" / "weights" / "best.pt"      # the detector that drafts the boxes
IMGSZ = 640                      # MUST match the size the model was trained at (960 for the larger model)

# --- drafting --------------------------------------------------------------------------------------------------
CONF = 0.20                      # low on purpose: reviewers delete wrong boxes faster than they draw missing ones
IOU = 0.5
MAX_DET = 20
ROLES_TO_DRAFT = ["pseudo-label + correct (training)", "pseudo-label + correct (validation)"]   # the HAND-BOX/test role is excluded
SKIP_IF_ALREADY_BOXED = True     # skip images that already have a boxed twin (column boxed_already in the CSV)
MAX_IMAGES = None                # e.g. 150 to start with a smaller first batch

# --- review (Windows folder, so Windows tools can open it) -----------------------------------------------------------
REVIEWERS = ["Bautista", "Pineda", "Valdez"]                    # one folder each; images are split evenly
REVIEW_DIR = Path("/mnt/c/Users/Bautista/Documents/PrecisionSkin_EczemaReview")

# --- final source folder (picked up by the audit notebook) -----------------------------------------------------------
FINAL_SOURCE_DIR = Path("/mnt/c/Users/Bautista/Downloads/Localization Training Datasets/eczema_corrected.yolov8")
APPLY_FINAL = False              # SAFETY SWITCH: while False, Stage 6 only shows what it would write
INCLUDE_EMPTY_AS_BACKGROUND = False   # a reviewed image with NO boxes: False = leave it out, True = keep as a no-eczema example

assert CANDIDATES_CSV.exists(), "candidates CSV not found: %s" % CANDIDATES_CSV
print("model :", MODEL_PATH, "| exists:", MODEL_PATH.exists())
print("review:", REVIEW_DIR)

## Stage 1 - Choose the images and split them between reviewers

In [ ]:
cand = pd.read_csv(CANDIDATES_CSV)
print("candidates:", len(cand))
cand = cand[cand.role_for_detector.isin(ROLES_TO_DRAFT)]
print("after role filter (classifier TEST images removed):", len(cand))
if SKIP_IF_ALREADY_BOXED and "boxed_already" in cand.columns:
    n0 = len(cand); cand = cand[~cand.boxed_already.astype(bool)]
    print("after skipping images that already have a boxed twin:", len(cand), "(skipped %d)" % (n0 - len(cand)))
cand["src_path"] = [p.replace("C:/", "/mnt/c/") for p in cand.local_path_windows]
missing = [p for p in cand.src_path if not Path(p).exists()]
print("source files missing:", len(missing))
cand = cand[cand.src_path.map(lambda p: Path(p).exists())].sort_values("filename").reset_index(drop=True)
if MAX_IMAGES:
    cand = cand.sample(n=min(MAX_IMAGES, len(cand)), random_state=1).sort_values("filename").reset_index(drop=True)
cand["reviewer"] = [REVIEWERS[i % len(REVIEWERS)] for i in range(len(cand))]
print("\nimages to draft:", len(cand)); print(cand.reviewer.value_counts().to_string())
print(cand.image_type.value_counts().to_string())

## Stage 2 - Draft the boxes with the detector
Runs the trained detector on every selected image and keeps **only the eczema class** (these images are eczema images). Writes, per reviewer:
`images/` (copies), `labels/` (YOLO draft boxes, class 0 = eczema) and `classes.txt`. A snapshot of the untouched drafts goes to `_drafts/` so the review can be measured.

In [ ]:
import torch
from ultralytics import YOLO
from tqdm.auto import tqdm

model = YOLO(str(MODEL_PATH))
names = model.names
ez_id = [k for k, v in names.items() if v == "eczema"][0]
print("model classes:", names, "| eczema id:", ez_id, "| GPU:", torch.cuda.is_available())

for rv in REVIEWERS:
    (REVIEW_DIR / rv / "images").mkdir(parents=True, exist_ok=True)
    (REVIEW_DIR / rv / "labels").mkdir(parents=True, exist_ok=True)
    (REVIEW_DIR / rv / "classes.txt").write_text("eczema\n")
(REVIEW_DIR / "_drafts").mkdir(parents=True, exist_ok=True)

drafts = {}
BATCH = 16
rows = list(cand.itertuples())
for s in tqdm(range(0, len(rows), BATCH), desc="drafting"):
    chunk = rows[s:s + BATCH]
    res = model.predict([r.src_path for r in chunk], imgsz=IMGSZ, conf=CONF, iou=IOU, max_det=MAX_DET, verbose=False, device=0 if torch.cuda.is_available() else "cpu")
    for r, o in zip(chunk, res):
        bx = []
        if o.boxes is not None and len(o.boxes):
            for (x0, y0, x1, y1), c, cf in zip(o.boxes.xyxyn.cpu().numpy(), o.boxes.cls.cpu().numpy(), o.boxes.conf.cpu().numpy()):
                if int(c) == ez_id:
                    bx.append((float((x0 + x1) / 2), float((y0 + y1) / 2), float(x1 - x0), float(y1 - y0), float(cf)))
        stem = Path(r.filename).stem
        drafts[r.filename] = bx
        txt = "".join("0 %.6f %.6f %.6f %.6f\n" % b[:4] for b in bx)
        dst_img = REVIEW_DIR / r.reviewer / "images" / r.filename
        if not dst_img.exists():
            shutil.copy2(r.src_path, dst_img)
        (REVIEW_DIR / r.reviewer / "labels" / (stem + ".txt")).write_text(txt)
        (REVIEW_DIR / "_drafts" / (stem + ".txt")).write_text(txt)

cand.to_csv(REVIEW_DIR / "review_assignments.csv", index=False)
json.dump({"drafted_at": time.time(), "model": str(MODEL_PATH), "conf": CONF, "imgsz": IMGSZ}, open(REVIEW_DIR / "_drafts" / "meta.json", "w"))
nb = np.array([len(v) for v in drafts.values()])
print("drafted %d images | with >=1 box: %d | mean boxes/image %.2f | images with NO draft box: %d" % (len(nb), (nb > 0).sum(), nb.mean(), (nb == 0).sum()))
print("review folders are in:", REVIEW_DIR)

## Stage 3 - Review (done by people, outside this notebook)
Each reviewer opens **their own folder** `PrecisionSkin_EczemaReview\<name>\` in a box editor. The simplest free tool is **LabelImg**
(`pip install labelImg` in a Windows Python 3.9-3.11, then run `labelImg`). Settings: **Open Dir** = `images`, **Change Save Dir** = `labels`, switch the format button to **YOLO**.

| Key | Action |
|---|---|
| `D` / `A` | next / previous image |
| `W` | draw a box |
| `Del` | delete the selected box |
| `Ctrl+S` | **save (required for EVERY image you accept, even if you changed nothing)** |

**Rules (agree on them once, write them in the paper)**
1. Keep a box only if it sits on **visible eczema** (red/scaly/dry inflamed skin). Delete boxes on normal skin, hair, eyes, lips, background.
2. Add a box for every clearly visible eczema patch the draft missed. Boxes may be loose; do not chase exact edges (eczema edges are soft).
3. One patch = one box. Merge boxes that cover the same patch.
4. If you cannot see eczema, or it is not a face, delete all boxes and save (the image will be left out).
5. **An image counts as reviewed only if you saved it** (the file's save time is checked). Unsaved images are ignored.

Any other box editor that reads/writes YOLO `.txt` (CVAT, Label Studio, Roboflow) works too: just keep the same folder layout and one class `eczema`.

## Stage 4 - Progress  (re-run any time)

In [ ]:
drafts_dir = REVIEW_DIR / "_drafts"
t_draft = json.load(open(drafts_dir / "meta.json"))["drafted_at"]
asg = pd.read_csv(REVIEW_DIR / "review_assignments.csv")

def read_boxes(p):
    if not Path(p).exists():
        return []
    out = []
    for ln in Path(p).read_text().splitlines():
        t = ln.split()
        if len(t) == 5:
            out.append(tuple(float(x) for x in t[1:]))
    return out

rows = []
for r in asg.itertuples():
    stem = Path(r.filename).stem
    lp = REVIEW_DIR / r.reviewer / "labels" / (stem + ".txt")
    saved = lp.exists() and lp.stat().st_mtime > t_draft + 1
    fin, dr = read_boxes(lp), read_boxes(drafts_dir / (stem + ".txt"))
    rows.append({"filename": r.filename, "reviewer": r.reviewer, "reviewed": bool(saved), "n_draft": len(dr), "n_final": len(fin), "changed": fin != dr})
prog = pd.DataFrame(rows)
print("REVIEWED (saved by a person): %d / %d  (%.0f%%)" % (prog.reviewed.sum(), len(prog), 100 * prog.reviewed.mean()))
print(prog.groupby("reviewer").agg(images=("filename", "size"), reviewed=("reviewed", "sum"), changed=("changed", "sum")).to_string())

## Stage 5 - How good were the drafts?  (a number for the paper)
Compares each reviewed image's **draft** with the **corrected** boxes (greedy IoU matching at 0.5). `draft precision` = share of draft boxes the reviewers kept;
`draft recall` = share of final boxes the detector had found. Only images saved by a reviewer are used.

In [ ]:
def iou(a, b):
    ax0, ay0, ax1, ay1 = a[0] - a[2] / 2, a[1] - a[3] / 2, a[0] + a[2] / 2, a[1] + a[3] / 2
    bx0, by0, bx1, by1 = b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
    iw, ih = max(0.0, min(ax1, bx1) - max(ax0, bx0)), max(0.0, min(ay1, by1) - max(ay0, by0))
    inter = iw * ih
    u = a[2] * a[3] + b[2] * b[3] - inter
    return inter / u if u > 0 else 0.0

def match(draft, final, thr=0.5):
    used, tp = set(), 0
    for d in draft:
        best, bj = 0.0, None
        for j, f in enumerate(final):
            if j in used:
                continue
            v = iou(d, f)
            if v > best:
                best, bj = v, j
        if bj is not None and best >= thr:
            used.add(bj); tp += 1
    return tp

tp = nd = nf = 0
rev = prog[prog.reviewed]
for r in rev.itertuples():
    stem = Path(r.filename).stem
    dr = read_boxes(drafts_dir / (stem + ".txt")); fin = read_boxes(REVIEW_DIR / r.reviewer / "labels" / (stem + ".txt"))
    tp += match(dr, fin); nd += len(dr); nf += len(fin)
if len(rev):
    print("reviewed images: %d | draft boxes: %d | final boxes: %d" % (len(rev), nd, nf))
    print("draft PRECISION (draft boxes kept):          %.1f%%" % (100 * tp / max(1, nd)))
    print("draft RECALL (final boxes the detector had): %.1f%%" % (100 * tp / max(1, nf)))
    print("images left with NO box after review:", int((rev.n_final == 0).sum()))
else:
    print("nothing reviewed yet")

## Stage 6 - Build the new source folder
Takes **reviewed** images that still have at least one box and writes them as a YOLO source folder `eczema_corrected.yolov8` (single class `eczema`) next to your other datasets,
so the audit notebook picks it up. Dry-run until `APPLY_FINAL = True`. After applying: re-run the audit notebook (add this source to `SOURCE_SITE_DECLARATION` as facial
and nothing else needs changing), then the export notebook with `DATASET_VERSION = "v2"`.

In [ ]:
ok = prog[prog.reviewed & ((prog.n_final > 0) | INCLUDE_EMPTY_AS_BACKGROUND)]
print("images to include:", len(ok), "| boxes:", int(ok.n_final.sum()))
print("left out: not reviewed %d | reviewed but no eczema visible %d" % ((~prog.reviewed).sum(), int((prog.reviewed & (prog.n_final == 0)).sum() if not INCLUDE_EMPTY_AS_BACKGROUND else 0)))
if not APPLY_FINAL:
    print("\nDRY-RUN: nothing written. Set APPLY_FINAL = True (Stage 0), re-run Stage 0, then this cell.")
else:
    assert not FINAL_SOURCE_DIR.exists() or not any(FINAL_SOURCE_DIR.iterdir()), "FINAL_SOURCE_DIR already has files; move it away first"
    (FINAL_SOURCE_DIR / "train" / "images").mkdir(parents=True, exist_ok=True); (FINAL_SOURCE_DIR / "train" / "labels").mkdir(parents=True, exist_ok=True)
    for r in ok.itertuples():
        stem = Path(r.filename).stem
        shutil.copy2(REVIEW_DIR / r.reviewer / "images" / r.filename, FINAL_SOURCE_DIR / "train" / "images" / r.filename)
        shutil.copy2(REVIEW_DIR / r.reviewer / "labels" / (stem + ".txt"), FINAL_SOURCE_DIR / "train" / "labels" / (stem + ".txt"))
    (FINAL_SOURCE_DIR / "data.yaml").write_text("train: ../train/images\nval: ../train/images\nnc: 1\nnames: ['Eczema']\n")
    (FINAL_SOURCE_DIR / "ORIGIN.txt").write_text("Boxes drafted by the PrecisionSkin detector (%s) and corrected by human reviewers.\nOrigin label for the paper: machine-drafted, human-corrected.\n" % MODEL_PATH.name)
    print("written:", FINAL_SOURCE_DIR, "| images:", len(ok))
    print("NEXT: re-run the audit notebook (new source appears automatically), then the export notebook as DATASET_VERSION = v2")